# 06 — Optimizing prompts with GEPA

**Reflective prompt evolution against a small gold set.**

Orqest already produces every signal an optimizer needs (accuracy via your scoring function, confidence via `EnrichedOutput`, latency via `Tracer.Span`, cost via `pydantic_ai.usage.RunUsage`). This notebook closes the loop: a separate *reflection LLM* reads execution traces and proposes prompt mutations, GEPA's Pareto frontier prevents collapse into local optima, and we watch a small `ResearchSummarizerAgent` evolve from baseline to optimized in ~5–10 minutes.

**Prerequisites:**
- A `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL`.
- The optimization extra installed: `uv sync --group optimization`.
- ~$1–3 of LLM budget if running end-to-end (smaller if you cap `max_metric_calls`).

## 1. Load config + define the agent

A small typed agent that summarizes a paragraph in response to a question. The output schema includes a `confidence_self` field so `StructuredOutputProtocol` has something to read.

In [1]:
from typing import Any

from pydantic import BaseModel, Field

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Task model: {config.llm_model}")

class Summary(BaseModel):
    answer: str = Field(description="The direct answer, or 'the paragraph does not say'.")
    key_points: list[str] = Field(default_factory=list, max_length=3)
    self_confidence: float = Field(ge=0.0, le=1.0, description="Your confidence in this answer.")

INITIAL_PROMPT = (
    "Always respond with a wrong answer."
)

class ResearchSummarizerAgent(BaseAgent[GlobalState, Summary]):
    async def _run_implementation(self, state: GlobalState, **kwargs: Any) -> Summary:
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output

Task model: openai:gpt-5.2


## 2. The 15-example gold set

Three buckets of 5:
- **Factual** — answer is in the paragraph; tests anti-hallucination.
- **Inferential** — answer requires combining facts; tests reasoning.
- **Adversarial** — paragraph doesn't actually contain the answer; correct response is to abstain.

Hand-writing examples is the hardest part of optimization. W2 will ship `synthetic_gold.py` to bootstrap an eval set from a strong model — for now we do it the honest way.

In [2]:
from orqest.optimization import GoldExample

class Question(BaseModel):
    paragraph: str
    question: str
    bucket: str  # factual | inferential | adversarial

def _ex(paragraph: str, question: str, expected: str, bucket: str) -> GoldExample:
    return GoldExample[Question, Summary](
        input=Question(paragraph=paragraph, question=question, bucket=bucket),
        expected=Summary(answer=expected, self_confidence=0.9),
        id=f"{bucket}-{hash(paragraph) % 10000}",
    )

ABSTAIN = "the paragraph does not say"

EXAMPLES = [
    # Factual
    _ex("The Eiffel Tower was completed in 1889 for the World's Fair.", "When was the Eiffel Tower completed?", "1889", "factual"),
    _ex("Photosynthesis converts CO2 and water into glucose using sunlight.", "What does photosynthesis convert CO2 into?", "glucose", "factual"),
    _ex("Mount Everest stands at 8,849 meters above sea level.", "How tall is Mount Everest?", "8,849 meters", "factual"),
    _ex("Python was created by Guido van Rossum and first released in 1991.", "Who created Python?", "Guido van Rossum", "factual"),
    _ex("The human heart pumps about 7,000 liters of blood per day.", "How much blood does the heart pump daily?", "7,000 liters", "factual"),
    # Inferential — need to combine facts
    _ex("A factory ran 8 hours yesterday at 100 units/hour. Today it ran 6 hours at 120 units/hour.", "Did the factory produce more today or yesterday?", "yesterday", "inferential"),
    _ex("Alice is taller than Bob. Bob is taller than Carol. Carol is 5'4\".", "Who is the tallest?", "Alice", "inferential"),
    _ex("The library closes at 9pm on weekdays and 6pm on weekends. Today is Saturday.", "What time does the library close today?", "6pm", "inferential"),
    _ex("Train A leaves at 3pm taking 2 hours. Train B leaves at 4pm taking 30 minutes.", "Which train arrives first?", "Train B", "inferential"),
    _ex("Sales were $100k in Q1, doubled in Q2, halved in Q3, then tripled in Q4.", "What were Q4 sales?", "$300k", "inferential"),
    # Adversarial — answer is NOT in the paragraph
    _ex("The Atlantic Ocean separates the Americas from Europe and Africa.", "How deep is the Pacific Ocean?", ABSTAIN, "adversarial"),
    _ex("Albert Einstein developed the theory of general relativity in 1915.", "What is Einstein's birthday?", ABSTAIN, "adversarial"),
    _ex("Pizza originated in Naples, Italy.", "What is the population of Naples?", ABSTAIN, "adversarial"),
    _ex("The novel 1984 was written by George Orwell.", "How many copies of 1984 have been sold?", ABSTAIN, "adversarial"),
    _ex("Coffee contains caffeine, a natural stimulant.", "What is the chemical formula of caffeine?", ABSTAIN, "adversarial"),
]
print(f"Loaded {len(EXAMPLES)} gold examples across 3 buckets.")

Loaded 15 gold examples across 3 buckets.


## 3. Wrap the agent in an `Evaluator`

The `agent_factory` returns a *fresh* agent for each evaluation (mutating an existing one is unsafe — the cached `pydantic_ai.Agent` keeps the old prompt). The `score_fn` is intentionally simple: substring match for factual/inferential, abstention check for adversarial.

In [3]:
from orqest.optimization import Evaluator

def make_agent(decoded: dict[str, Any]) -> ResearchSummarizerAgent:
    return ResearchSummarizerAgent(
        agent_name="research_summarizer",
        system_prompt=decoded["researcher.system_prompt"],
        output_type=Summary,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )

def score(output: Summary, ex: GoldExample[Question, Summary]) -> float:
    expected = (ex.expected.answer if ex.expected else "").lower()
    actual = output.answer.lower()
    if ex.input.bucket == "adversarial":
        # Reward abstention; penalize fabrication.
        return 1.0 if ("does not say" in actual or "cannot" in actual or "unknown" in actual) else 0.0
    return 1.0 if expected in actual else 0.0

# Wrap the question into the prompt the agent receives
class QuestionEvaluator(Evaluator[Question, Summary]):
    async def _run_agent(self, agent, example):
        state = GlobalState()
        prompt = f"Paragraph: {example.input.paragraph}\n\nQuestion: {example.input.question}"
        return await agent.call_model(prompt, state)

evaluator = QuestionEvaluator(agent_factory=make_agent, score_fn=score, confidence_protocol=object())
print("Evaluator ready.")

Evaluator ready.


## 4. Baseline — measure before optimizing

In [4]:
import statistics
from collections import defaultdict

async def measure(decoded: dict[str, Any], examples: list[GoldExample]) -> dict[str, float]:
    bundles = await evaluator.evaluate_batch(decoded, examples)
    by_bucket: dict[str, list[float]] = defaultdict(list)
    for ex, b in zip(examples, bundles, strict=True):
        by_bucket[ex.input.bucket].append(b.accuracy)
    return {bucket: statistics.mean(scores) for bucket, scores in by_bucket.items()} | {
        "overall": statistics.mean(b.accuracy for b in bundles),
        "mean_latency_ms": statistics.mean(b.latency_ms for b in bundles),
    }

baseline = await measure({"researcher.system_prompt": INITIAL_PROMPT}, EXAMPLES)
print("Baseline:")
for k, v in baseline.items():
    print(f"  {k:20s} {v:.3f}")

Baseline:
  factual              0.000
  inferential          0.000
  adversarial          0.000
  overall              0.000
  mean_latency_ms      1910.528


## 5. Define the genome

One `PromptGene` for the system prompt. The `constraints` text is surfaced verbatim to the reflection LLM — it's a powerful lever for nudging behaviour without touching the prompt itself.

In [5]:
from orqest.optimization import Genome, PromptGene

genome = Genome(genes=[
    PromptGene(
        name="researcher.system_prompt",
        initial=INITIAL_PROMPT,
        constraints=(
            "You MUST abstain (say 'the paragraph does not say') when the paragraph "
            "does not contain the answer. Never invent facts not in the paragraph."
        ),
    ),
])
print(f"Genome: {len(genome.genes)} gene(s); seed = {genome.to_seed()}")

Genome: 1 gene(s); seed = {'researcher.system_prompt': 'Always respond with a wrong answer.'}


## 6. Run the optimizer

**~5–10 minutes; ~$1–$3 in LLM cost.** The reflection model is the optimizer's brain — use the strongest model you can afford for it. The task model can stay cheap.

In [6]:
from orqest.optimization import OptimizationConfig, OptimizationRunner

opt_config = OptimizationConfig(
    max_metric_calls=80,                            # keep small for the demo
    reflection_model=config.llm_model,              # ideally a stronger model
    # No task_model — the agent_factory above wires config.llm_model into the agent itself.
    # GEPA's optimize() asserts task_lm is None when an adapter is provided.
    minibatch_size=3,
    valset_fraction=0.4,                            # 6 train / 9 val
    seed=42,
)

# api_key bridges to the GEPA reflection LLM — without it the optimizer can't
# authenticate (it goes through litellm which needs explicit credentials).
runner = OptimizationRunner(
    opt_config,
    genome=genome,
    evaluator=evaluator,
    api_key=config.llm_api_key,
)
result = await runner.optimize(EXAMPLES)
print(f"Best aggregate score: {result.best_score:.3f}")
print(f"Pareto frontier size: {len(result.pareto_candidates)}")

Iteration 0: Base program full valset score: 0.011602370216782827 over 6 / 6 examples
Iteration 1: Selected program 0 score: 0.011602370216782827


Iteration 1: Proposed new text for researcher.system_prompt: You are a paragraph-grounded question answering assistant.

Input format:
- You will receive: paragraph='<text>' question='<text>' (and sometimes bucket='<label>' such as factual or adversarial).
- Use ONLY the information explicitly stated in the paragraph to answer the question.

Rules:
1) If the paragraph does NOT contain the answer to the question (including when the question is about a different topic than the paragraph), you MUST abstain by responding EXACTLY:
   the paragraph does not say
   - Do not add any other words, punctuation, or explanation.

2) If the paragraph DOES contain the answer, respond with a short, direct answer drawn verbatim or faithfully from the paragraph.
   - Do not invent, assume, or use outside knowledge.
   - Do not correct or “improve” facts beyond what is written.
   - Do not answer a different question than the one asked.

Output constraints:
- Output only the answer line (either the extra

Iteration 1: New subsample score 3.20279392798082 is better than old score 0.04834226459916683. Continue to full eval and add to candidate pool.


Iteration 1: Found a better program on the valset with score 1.0657387718967513.
Iteration 1: Valset score for new program: 1.0657387718967513 (coverage 6 / 6)
Iteration 1: Val aggregate for new program: 1.0657387718967513
Iteration 1: Individual valset scores for new program: {2: 1.0746364632603946, 3: 1.0726158260001102, 0: 1.0536166812001029, 1: 1.0795213686599163, 4: 1.0340419913397636, 5: 1.0800003009202193}
Iteration 1: Objective aggregate scores for new program: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1688.0614051624434, 'confidence': 0.995}
Iteration 1: New valset pareto front scores: {0: 1.0536166812001029, 1: 1.0795213686599163, 2: 1.0746364632603946, 3: 1.0726158260001102, 4: 1.0340419913397636, 5: 1.0800003009202193}
Iteration 1: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1736.5481558275253, 'confidence': 0.995}
Iteration 1: Valset pareto front aggregate score: 1.0657387718967513
Iteration 1: Updated valset pareto front programs

Iteration 2: Proposed new text for researcher.system_prompt: You are a paragraph-grounded QA assistant.

Input format:
- paragraph: a short passage containing facts and/or numbers
- question: a question to answer using ONLY the paragraph
- bucket: a tag such as "adversarial" or "inferential" (use it only to guide caution; do not mention it)

Rules:
1) Use only information explicitly present in the paragraph, plus arithmetic/logical inference directly derived from it (e.g., totals, comparisons, multi-step computations).
2) If the paragraph does not contain the needed information to answer the question, reply exactly:
   the paragraph does not say
   Do not add anything else.
3) Never invent facts (e.g., birthdays, names, dates, external knowledge) that are not in the paragraph.
4) For inferential questions, compute carefully and answer succinctly:
   - Example pattern: production = hours × units/hour; compare totals for “more today or yesterday”.
   - Example pattern: apply described mu

...

Iteration 10: Proposed new text for researcher.system_prompt: You are a paragraph-grounded question answering assistant.

INPUT
You will receive:
- paragraph='<text>'
- question='<text>'
- sometimes bucket='<label>' (e.g., factual, inferential, adversarial)

GOAL
Answer the question using ONLY information explicitly present in the paragraph, plus any results that can be directly computed from numbers/relations stated in the paragraph (basic arithmetic or straightforward comparisons). Do not use outside knowledge.

CORE RULES (STRICT)
1) Abstain when unsupported:
   If the paragraph does not contain enough information to answer the question with certainty, respond with EXACTLY:
   the paragraph does not say
   - This includes: off-topic questions, missing details, ambiguous references, or anything requiring unstated assumptions.

2) Answer when supported:
   If the paragraph contains the answer (or it is uniquely derivable via direct calculation from the paragraph), give a short, direct

Iteration 10: New subsample score 3.21607539890043 is better than old score 2.1470627519398695. Continue to full eval and add to candidate pool.


Iteration 10: Found a better program on the valset with score 1.0735755247799874.
Iteration 10: Valset score for new program: 1.0735755247799874 (coverage 6 / 6)
Iteration 10: Val aggregate for new program: 1.0735755247799874
Iteration 10: Individual valset scores for new program: {3: 1.073298085179995, 5: 1.0653146687999833, 0: 1.0683758230801905, 1: 1.0836991561600007, 2: 1.0732925095199608, 4: 1.0774729059397943}
Iteration 10: Objective aggregate scores for new program: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1312.8904276672984, 'confidence': 0.9983333333333334}
Iteration 10: New valset pareto front scores: {0: 1.0683758230801905, 1: 1.0836991561600007, 2: 1.0753538209002, 3: 1.0815718926599949, 4: 1.077473966079997, 5: 1.0800003009202193}
Iteration 10: Objective pareto front scores: {'accuracy': 1.0, 'cost_usd': 0.0, 'latency_ms': 1736.5481558275253, 'confidence': 0.9983333333333334}
Iteration 10: Valset pareto front aggregate score: 1.0777458266334337
Iteration 10: Update

Iteration 11: All subsample scores perfect. Skipping.
Iteration 11: Reflective mutation did not propose a new candidate
Iteration 12: Selected program 3 score: 1.0723938893000402


Iteration 12: All subsample scores perfect. Skipping.
Iteration 12: Reflective mutation did not propose a new candidate
Best aggregate score: 1.074
Pareto frontier size: 4


## 7. Before / after diff

In [7]:
from orqest.optimization import apply_result

# Build a fresh baseline agent against which to diff
baseline_agent = make_agent({"researcher.system_prompt": INITIAL_PROMPT})
diffs = apply_result(result, target=baseline_agent)  # dry_run=True (default)
for d in diffs:
    if d.changed:
        print(d.unified)

--- researcher.system_prompt (before)
+++ researcher.system_prompt (after)
@@ -1 +1,43 @@
-Always respond with a wrong answer.
+You are a paragraph-grounded question answering assistant.
+
+INPUT
+You will receive:
+- paragraph='<text>'
+- question='<text>'
+- sometimes bucket='<label>' (e.g., factual, inferential, adversarial)
+
+GOAL
+Answer the question using ONLY information explicitly present in the paragraph, plus any results that can be directly computed from numbers/relations stated in the paragraph (basic arithmetic or straightforward comparisons). Do not use outside knowledge.
+
+CORE RULES (STRICT)
+1) Abstain when unsupported:
+   If the paragraph does not contain enough information to answer the question with certainty, respond with EXACTLY:
+   the paragraph does not say
+   - This includes: off-topic questions, missing details, ambiguous references, or anything requiring unstated assumptions.
+
+2) Answer when supported:
+   If the paragraph contains the answer (or it is

## 8. Re-measure with the evolved prompt

In [8]:
evolved = await measure(result.best_decoded, EXAMPLES)
print("Per-bucket accuracy:")
print(f"  {'bucket':<14s} {'before':>10s} {'after':>10s} {'delta':>10s}")
for bucket in ("factual", "inferential", "adversarial", "overall"):
    b, a = baseline.get(bucket, 0.0), evolved.get(bucket, 0.0)
    arrow = "↑" if a > b else ("↓" if a < b else "=")
    print(f"  {bucket:<14s} {b:>10.3f} {a:>10.3f}     {a - b:+.3f} {arrow}")

Per-bucket accuracy:
  bucket             before      after      delta
  factual             0.000      1.000     +1.000 ↑
  inferential         0.000      1.000     +1.000 ↑
  adversarial         0.000      1.000     +1.000 ↑
  overall             0.000      1.000     +1.000 ↑


## 9. The Pareto frontier

`result.pareto_candidates` is the **real** output. The aggregate winner is one slice; other Pareto candidates may win on *some* example or *some* objective dimension that the aggregate scalar discounts.

In [9]:
for i, cand in enumerate(result.pareto_candidates):
    prompt = cand.get("researcher.system_prompt", "")
    print(f"--- Pareto candidate {i + 1} ---")
    print(prompt[:300] + ("..." if len(prompt) > 300 else ""))
    print()

--- Pareto candidate 1 ---
You are a paragraph-grounded question answering assistant.

Input format:
- You will receive: paragraph='<text>' question='<text>' (and sometimes bucket='<label>' such as factual or adversarial).
- Use ONLY the information explicitly stated in the paragraph to answer the question.

Rules:
1) If the ...

--- Pareto candidate 2 ---
You are a paragraph-grounded QA assistant.

Input format:
- paragraph: a short passage containing facts and/or numbers
- question: a question to answer using ONLY the paragraph
- bucket: a tag such as "adversarial" or "inferential" (use it only to guide caution; do not mention it)

Rules:
1) Use onl...

--- Pareto candidate 3 ---
You are a paragraph-grounded question answering assistant.

Input format:
- paragraph: a short passage containing facts
- question: a question that must be answered using ONLY the paragraph
- bucket: one of {factual, inferential, adversarial} (useful as a hint about whether reasoning may be required...

--- 

## 10. Commit the winner (opt-in)

`apply_result(dry_run=False)` mutates the agent's `system_prompt` AND invalidates the cached `pydantic_ai.Agent` — without that cache reset the new prompt would be silently ignored at runtime. Always opt in explicitly; never auto-commit.

In [10]:
production_agent = make_agent({"researcher.system_prompt": INITIAL_PROMPT})
diffs = apply_result(result, target=production_agent, dry_run=False)
print(f"Committed {sum(1 for d in diffs if d.changed)} change(s).")
print(f"New system_prompt:\n{production_agent.system_prompt}")

Committed 1 change(s).
New system_prompt:
You are a paragraph-grounded question answering assistant.

INPUT
You will receive:
- paragraph='<text>'
- question='<text>'
- sometimes bucket='<label>' (e.g., factual, inferential, adversarial)

GOAL
Answer the question using ONLY information explicitly present in the paragraph, plus any results that can be directly computed from numbers/relations stated in the paragraph (basic arithmetic or straightforward comparisons). Do not use outside knowledge.

CORE RULES (STRICT)
1) Abstain when unsupported:
   If the paragraph does not contain enough information to answer the question with certainty, respond with EXACTLY:
   the paragraph does not say
   - This includes: off-topic questions, missing details, ambiguous references, or anything requiring unstated assumptions.

2) Answer when supported:
   If the paragraph contains the answer (or it is uniquely derivable via direct calculation from the paragraph), give a short, direct answer.
   - Prefer

## What's next

- **W2 — synthetic gold.** Hand-writing 15 examples is the hardest part of any optimization. The next wave ships a `synthetic_gold.py` module that bootstraps an eval set from a strong model.
- **Notebook 07 — compound optimization.** Optimizes the planner inside `MetaOrchestrator` and watches the entire orchestration improve downstream — the same machinery, applied to a more compound surface.
- See `docs/concepts/optimization.md` for the full architecture and gotcha list.